# fedOPD · Stage 6 — Agentic capacity planning (narrative notebook)

Turns the federated demand forecasts (Stage 5) into capacity/allocation decisions. This notebook is
the readable narrative; the reusable building blocks live in `src/fedopd/*.py` (imported below), and
the **decision policy itself is written out inline here** so it can be inspected end to end.

**Flow state (from the same raw files, identical filters):** demand `Part_3`, throughput `Part_1A+1B`,
backlog `Part_2`, and the >18-week waiters (`Part_2` bands ≥18wk) — table `reports/tables/rtt_flow.parquet`.

**Decision model (Little's law):** mean wait (months) ≈ backlog / throughput. The 18-week standard
(4.14 months) ⇒ sustainable backlog `B* = 4.14 · D_hat`. Required throughput and the capacity gap:
`C_req = D_hat + max(0, B − B*)/12`, `gap = C_req − C_base`.

## 1 · Load the flow state and the federated forecasts

In [1]:
import sys, json
import numpy as np, pandas as pd
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/"CLAUDE.md").exists())
sys.path.insert(0, str(ROOT))
from src.fedopd import data as fdata, fed as ffed   # reuse the validated data + FL building blocks

flow = pd.read_parquet(ROOT/"reports"/"tables"/"rtt_flow.parquet")
print("flow table:", flow.shape, "| cols:", list(flow.columns))

# 1-month-ahead demand forecast per (trust, dept, test-month) from the deployed model (fedavg + personalise)
ds = fdata.build_all(); cd = fdata.per_client(ds)
glob,_ = ffed.run_federated(cd, ds["n_features"], "delta", rounds=40, seed=0)
models = ffed.personalise(glob, cd, ds["n_features"], "delta", ft_epochs=3, seed=0)
pred = ffed.predict_per_client(models, ds["te_df"], ds["Xte"], lag1=ds["n_lags"]-1)
fc = ds["te_df"].copy()
fc["D_hat"] = np.maximum(0.0, pred*fc["sd"].values + fc["mean"].values)
fc["date"] = pd.to_datetime(fc["month"])
fc = fc[["branch","dept","date","actual","D_hat"]].rename(columns={"actual":"D_actual"})
print("forecasts:", fc.shape, "| test months:", sorted(fc.date.dt.strftime("%Y-%m").unique())[:3], "...")

flow table: (231508, 8) | cols: ['date', 'branch', 'icb', 'dept', 'demand', 'completed', 'backlog', 'breach18']


forecasts: (18191, 5) | test months: ['2025-04', '2025-05', '2025-06'] ...


## 2 · Validate the flow model
The whole plan rests on the identity **backlog(t) ≈ backlog(t−1) + demand − completed**. Check it
nationally before trusting any per-service decision.

In [2]:
nat = flow.groupby("date")[["demand","completed","backlog"]].sum()
pred_b = (nat["backlog"].shift(1) + nat["demand"] - nat["completed"]).iloc[1:]
err = ((pred_b - nat["backlog"].iloc[1:]) / nat["backlog"].iloc[1:]).abs()
print(f"flow identity — national median abs error: {err.median():.1%}  (clock pauses/removals cause the rest)")

flow identity — national median abs error: 3.8%  (clock pauses/removals cause the rest)


## 3 · The decision policy (written out inline)
For every service×month: required throughput, capacity gap, and a RAG status from the implied wait.

In [3]:
WAIT_TARGET = 18/4.345          # 18-week standard in months (~4.14)
HORIZON = 12                    # months to burn the backlog excess to target

flow = flow.sort_values(["branch","dept","date"])
g = flow.groupby(["branch","dept"], group_keys=False)
flow["B_prev"]      = g["backlog"].shift(1)
flow["breach_prev"] = g["breach18"].shift(1)
flow["C_base"]      = g["completed"].transform(lambda s: s.shift(1).rolling(12, min_periods=3).mean())

dec = fc.merge(flow[["branch","dept","date","backlog","completed","breach18",
                     "B_prev","breach_prev","C_base","icb"]], on=["branch","dept","date"], how="left")
dec = dec.dropna(subset=["B_prev","C_base"]); dec = dec[dec["C_base"] > 0].copy()

B_star        = WAIT_TARGET * dec["D_hat"]                       # sustainable backlog (Little's law)
dec["C_req"]  = dec["D_hat"] + np.maximum(0.0, dec["B_prev"] - B_star)/HORIZON
dec["gap"]    = dec["C_req"] - dec["C_base"]                     # >0 => allocate more capacity
dec["wait_wk"]= dec["B_prev"] / dec["C_base"] * 4.345
dec["rag"]    = np.where(dec["wait_wk"]>=18,"RED", np.where(dec["wait_wk"]>=12,"AMBER","GREEN"))
print("decisions computed for", len(dec), "service-months")
dec[["branch","dept","date","D_hat","B_prev","C_base","C_req","gap","wait_wk","rag"]].tail(3)

decisions computed for 18191 service-months


,branch,dept,date,D_hat,B_prev,C_base,C_req,gap,wait_wk,rag
18188,RYW,C_140,2026-01-01,1137.224187,7547.0,1233.166667,1373.543321,140.376654,26.591470,RED
18189,RYW,C_140,2026-02-01,1006.162039,7861.0,1242.333333,1313.893690,71.560357,27.493463,RED
18190,RYW,C_140,2026-03-01,857.738403,7370.0,1268.833333,1175.792847,-93.040486,25.237869,RED


## 4 · Decision snapshot (latest month) + ICB allocation ranking

In [4]:
latest = dec.date.max(); snap = dec[dec.date==latest]
print("decision month:", latest.date(), "| services:", len(snap))
print("RAG:", snap.rag.value_counts().to_dict())
print(f"national additional capacity needed: {int(snap['gap'].clip(lower=0).sum()):,} pathways/month")
icb = (snap.assign(pos=snap.gap.clip(lower=0)).groupby("icb")
          .agg(add_capacity=("pos","sum"), red=("rag",lambda s:(s=="RED").sum()))
          .sort_values("add_capacity", ascending=False))
print("\ntop ICBs by additional capacity needed:")
print(icb.head(8).assign(add_capacity=lambda d:d.add_capacity.astype(int)).to_string())

decision month: 2026-03-01 | services: 1509
RAG: {'RED': 969, 'AMBER': 322, 'GREEN': 218}
national additional capacity needed: 216,046 pathways/month

top ICBs by additional capacity needed:
     add_capacity  red
icb                   
QOP         12201   49
QH8         11654   15
QRV         11559   49
QKK         10355   28
QHG         10212   25
QE1          9882   47
QU9          9637   33
QHM          9626   40


## 5 · Validation — the RAG proxy vs the ground-truth wait bands
RAG comes from the Little's-law wait proxy (backlog/throughput). The actual >18-week share comes
independently from the `Part_2` wait bands. If RED carries the highest real share, the proxy agrees
with the ground truth (non-tautological).

In [5]:
d = dec.dropna(subset=["breach_prev","B_prev"]); d = d[d.B_prev>0]
share = (d.breach_prev/d.B_prev).clip(0,1)*100
print("median actual >18wk backlog share by RAG:")
print(share.groupby(d.rag).median().reindex(["GREEN","AMBER","RED"]).round(1).to_string())
rho = d.assign(net=d.D_actual-d.completed, dB=d.backlog-d.B_prev)[["net","dB"]].corr("spearman").iloc[0,1]
print(f"\nflow-model validity (per service-month): Spearman rho = {rho:.2f}")

median actual >18wk backlog share by RAG:
rag
GREEN    11.0
AMBER    27.0
RED      42.7

flow-model validity (per service-month): Spearman rho = 0.40


## 6 · Figures (inline) + persist the decisions

In [6]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, ax = plt.subplots(1,2, figsize=(12,3.6))
n = flow.groupby("date")[["demand","completed","backlog"]].sum()/1e6
ax[0].plot(n.index,n.demand,label="demand"); ax[0].plot(n.index,n.completed,label="completed")
ax2=ax[0].twinx(); ax2.plot(n.index,n.backlog,color="crimson",label="backlog"); ax2.set_ylabel("backlog (M)")
ax[0].legend(loc="upper left"); ax[0].set_title("national flow (M/mo)")
top=icb.head(8)[::-1]; ax[1].barh(top.index, top.add_capacity/1e3, color="#005EB8")
ax[1].set_title("ICB allocation priority (k pathways/mo)")
fig.tight_layout(); plt.show()
dec.to_parquet(ROOT/"results"/"agentic_decisions.parquet")
print("saved results/agentic_decisions.parquet")

saved results/agentic_decisions.parquet


## Summary
The agentic plane converts private federated forecasts into auditable capacity decisions: per-service
RAG + capacity gap, rolled up to an ICB allocation ranking, validated against the independent wait-band
data. Reusable code: `src/fedopd/{extract_flow,agentic,make_figures}.py`. Everything is CPU-only and
reproducible. Scope is NHS trusts (allocate); independents would be commissioned (a later extension).